# The chain rule of MatrixFunction

SymPy 1.14 has no undefined matrix-valued function.
[sympy_matrix_tools](https://github.com/JRF-2018/sympy_matrix_tools)
provides `MatrixFunction` and differentiates it by the chain rule.
This notebook shows the derivatives and checks them against direct
differentiation after substituting concrete functions.

In [ ]:
!pip install "sympy>=1.14" git+https://github.com/JRF-2018/sympy_matrix_tools

In [2]:
from sympy import *
from sympy_matrix_tools import *

In [3]:
t = Symbol("t")
s = Symbol("s")
X = MatrixSymbol("X", 2, 2)
Y = MatrixSymbol("Y", 2, 2)
A = MatrixSymbol("A", 2, 2)
D = MatrixSymbol("D", 2, 2)
F = MatrixFunction("F", 2, 2)
G = MatrixFunction("G", 2, 2)

## By a scalar

For a scalar argument, the derivative is the usual `Subs` of a
`Derivative`, but as a matrix expression.

In [4]:
F(t**2).diff(t)

(2*t)*Subs(Derivative(F(_xi_1), _xi_1), _xi_1, t**2)

For a matrix argument, the derivative by a scalar is represented by a
directional derivative:
$\frac{d}{dt} F(a(t)) = \frac{d}{ds} F(a(t) + s\, a'(t))\big|_{s=0}$.

In [5]:
F(t * X).diff(t)

Subs(Derivative(F(t*X + _s_1*X), _s_1), _s_1, 0)

Substituting a concrete function gives the expected result.

In [6]:
F(t * X).diff(t).subs(F, Lambda(D, D * A * D)).doit()

(2*t)*X*A*X

## By a matrix

By a matrix `X`, the derivative is an array expression whose indices
of `X` come first, as in SymPy's `derive_by_array`. Use `mat_diff`.

In [7]:
r = mat_diff(F(X * Y), X)
r

ArrayContraction(ArrayTensorProduct(Y, I, I, Subs(Derivative(F(_xi_1), _xi_1), _xi_1, X*Y)), (0, 5), (1, 7), (3, 6))

In [8]:
r.shape

(2, 2, 2, 2)

## Checking against direct differentiation

`check` substitutes concrete functions for the `MatrixFunction`s and
compares the derivative with the direct differentiation of the
explicit matrices.

In [9]:
def explicit (e):
    if hasattr(e, 'as_explicit'):
        e = e.as_explicit()
    return Array(e)

def check (expr, v, sigma):
    lhs = mat_diff(expr, v) if v.is_Matrix else expr.diff(v)
    rhs = expr
    for f, lam in sigma:
        lhs = lhs.subs(f, lam)
        rhs = rhs.subs(f, lam)
    lhs = explicit(lhs.doit())
    rhs = explicit(rhs.doit())
    rhs = derive_by_array(rhs, explicit(v)) if v.is_Matrix else rhs.diff(v)
    lhs, rhs = Array(lhs), Array(rhs)
    return lhs.shape == rhs.shape and \
        all(simplify(a - b) == 0 for a, b in
            zip(lhs.reshape(len(lhs)), rhs.reshape(len(rhs))))

LF = Lambda(D, D * A * D)
LG = Lambda(D, D.T * D + A)
cases = [
    ("F(X*Y) by X", F(X * Y), X, [(F, LF)]),
    ("F(G(X)) by X", F(G(X)), X, [(F, LF), (G, LG)]),
    ("A*F(X)*Y by X", A * F(X) * Y, X, [(F, LF)]),
    ("F(t*X) by t", F(t * X), t, [(F, LF)]),
    ("F(G(t*X)) by t", F(G(t * X)), t, [(F, LF), (G, LG)]),
]
for name, e, v, sigma in cases:
    print(name, check(e, v, sigma))
    assert check(e, v, sigma)

F(X*Y) by X True


F(G(X)) by X True


A*F(X)*Y by X True


F(t*X) by t True


F(G(t*X)) by t True


## A bug of SymPy 1.14

`F(X*Y).diff(X)` uses SymPy's `convert_array_to_matrix`, which
returns a wrong array of rank 4 in this case. `mat_diff` returns the
array expression as it is.

In [10]:
LFY = lambda e: Array(e.subs(F, LF).doit().as_explicit())
direct = derive_by_array((X * Y * A * X * Y).as_explicit(), X.as_explicit())
print("mat_diff:", (LFY(mat_diff(F(X * Y), X)) - direct).applyfunc(expand) == Array.zeros(2, 2, 2, 2))
print(".diff:   ", (LFY(F(X * Y).diff(X)) - direct).applyfunc(expand) == Array.zeros(2, 2, 2, 2))

mat_diff: True
.diff:    False


## Transposes

Importing sympy_matrix_tools also fixes `Transpose._eval_derivative`
of SymPy 1.14, which forgot to transpose.

In [11]:
x = MatrixFunction("x", 3, 1)
B = MatrixSymbol("B", 3, 3)
(x(t).T * B * x(t)).diff(t)

x(t).T*B*Derivative(x(t), t) + Derivative(x(t), t).T*B*x(t)